Requirements

In [ ]:
pip install transformers accelerate

In [ ]:
pip install pysentimiento

In [ ]:
pip install nltk

In [ ]:
pip install transformers[sentencepiece]

In [ ]:
pip install spacy

In [ ]:
pip install asent

Libraries

In [ ]:
import pandas as pd
from pysentimiento import create_analyzer
from transformers import pipeline
import nltk #comprobar si quitando esta linea funciona el codigo
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import spacy
import asent
import numpy as np
import glob
import matplotlib.pyplot as plt
import numpy
from sklearn.metrics import classification_report, confusion_matrix

Clasificar texto con pysentimiento

In [ ]:
def PysentimentClasificator(data:str, ColumnName:str):
  try:
    df = pd.read_excel(data, sheet_name='Sheet1')
    resultado={}
    analyzer = create_analyzer(task="sentiment", lang="es")
    for index, row in df.iterrows():
      out=analyzer.predict(row[ColumnName])#clasifica el texto
      resultado[index]=out.output
      print(index)
  except:
    return 'Failed'


  frame=pd.DataFrame( {'clasificacion_pysentimiento': resultado})   #crear nuevo dataframe con una columna de clasificacion
  frame.to_excel('score_pysentiment.xlsx')
  return 'Success'

Clasificando con VADE

Traducir texto a ingles

In [ ]:
#luego de instalar transformers[sentencepiece] reiniciar maquina y correr de nuevo

def TraduccionText(data:str, ColumnName:str):
  en_text={}
  df = pd.read_excel(data, sheet_name='Sheet1')
  #tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
  translator = pipeline("translation",
                      model="Helsinki-NLP/opus-mt-es-en") #modelo
  for index, row in df.iterrows():
    try:
      if len(row[ColumnName])>390:
        opinion=row[ColumnName][0:390]
        traduccion=translator(opinion)
      else:
        traduccion=translator(row[ColumnName])
      en_text[index]=traduccion[0]['translation_text']
      print(index)
    except:
      print("no se pudo ejecutar")

  frame=pd.DataFrame( {'EN_Opiniones': en_text})
  frame.to_excel('score_en.xlsx')
  return frame

In [ ]:
# Importamos las bibliotecas necesarias
def VaderClasificator(data:str, ColumnName:str):
  df=TraduccionText(data,ColumnName)
  nltk.download('vader_lexicon')
  vader_clasif={}
  #Creamos el analizador de sentimientos
  vader = SentimentIntensityAnalyzer()
  #Analizamos cada frase
  for index,row in df.iterrows():
    sentiment = vader.polarity_scores(row['EN_Opiniones'])
    print(index, '---',sentiment)
    if sentiment['compound']>0.3:
      vader_clasif[index]='POS'
    elif sentiment['compound']<0.3 and sentiment['compound']>0:
      vader_clasif[index]='NEU'
    else:
      vader_clasif[index]='NEG'

  frame=pd.DataFrame( {'Clasificacion_Vader': vader_clasif})
  #final=pd.concat([df,fr_vader], axis=1)
  frame.to_excel('score_vader.xlsx')

Clasificar con spacy

In [ ]:
def AsentClasificator(data:str, ColumnName:str, C=True):
  if C:
    df=TraduccionText(data,ColumnName)
  else:
      df = pd.read_excel('score_en.xlsx', sheet_name='Sheet1')

  asentimiento_clasif={}
  # load spacy pipeline
  nlp = spacy.blank('en')
  nlp.add_pipe('sentencizer')

  # add the rule-based sentiment model
  nlp.add_pipe('asent_en_v1')


  for index, row in df.iterrows():
      sentiment=nlp(row['EN_Opiniones'])#clasifica el texto
      asentimiento_clasif[index]=sentiment._.polarity   #me quede aqui
      print(asentimiento_clasif[index])
      print(index)
      if asentimiento_clasif[index].compound >0.2:
        asentimiento_clasif[index]='POS'
      elif asentimiento_clasif[index].compound<0.2 and asentimiento_clasif[index].compound>0 :
        asentimiento_clasif[index]='NEU'
      else:
        asentimiento_clasif[index]='NEG'

  frame=pd.DataFrame( {'Clasificacion_Asentiment': asentimiento_clasif})   #crear nuevo dataframe con una columna de clasificacion
  #final=pd.concat([df,fr_asent], axis=1)
  frame.to_excel('score_asent.xlsx')

Concadenar  todos los dataframe

Algoritmo de dos tuplas

In [ ]:
def interception(line1, line2):
    xdiff = (line1[0][0] - line1[1][0], line2[0][0] - line2[1][0])
    ydiff = (line1[0][1] - line1[1][1], line2[0][1] - line2[1][1])
    div = det(xdiff, ydiff)
    return div!=0 # no ocurre intercepcion


def det(a, b):
    return a[0] * b[1] - a[1] * b[0]

def line_intersection(line1, line2):
    xdiff = (line1[0][0] - line1[1][0], line2[0][0] - line2[1][0])
    ydiff = (line1[0][1] - line1[1][1], line2[0][1] - line2[1][1])

    div = det(xdiff, ydiff)

    d = (det(*line1), det(*line2))
    x = det(d, xdiff) / div
    y = det(d, ydiff) / div
    return x, y

def membership_triangle_function(value,line):
    m=(line[1][1]-line[0][1])/(line[1][0]-line[0][0])
    if value >=line[0][0] and value<=line[1][0] or value >=line[1][0] and value <=line[0][0]:
        if m>0:
            return (value - line[0][0])/(line[1][0]-line[0][0])
        else:
            return (line[1][0]-value)/(line[1][0]-line[0][0])
    else:
        return 0

def generate_fuzzy_set(n):
    set_fuzz={}
    interval=1/(n-1)
    x=0
    for i in range(0,n-1):
        if i ==0:
            line=[(x,1),(x+interval,0)]
            set_fuzz[i]=[line]
        else:
            line1=[(x-interval,0),(x,1)]
            line2=[(x,1),(x+interval,0)]
            set_fuzz[i]=[line1,line2]
        x+=interval

    line=[(x-interval,0),(x,1)]
    set_fuzz[n-1]=[line]

    return set_fuzz
def transform_to_fuzzy_set(value,linguistic_set):
    fuzzy_set=[]
    for set_point in linguistic_set.values():
        pair2=pair1=None
        enc=False
        for i in range(len(set_point)):
            for j in range(len(value)):
                if interception(set_point[i],value[j]):
                    x,y=line_intersection(set_point[i],value[j])
                    if y>0 and y<=1:
                        pair1=set_point[i]
                        pair2=value[j]
                        enc=True
        if enc:
            x,_=line_intersection(pair1,pair2)
            fuzzy_set.append(membership_triangle_function(x,pair1))
        else:
            fuzzy_set.append(0)

    return fuzzy_set

def fuzzy_set_2_tuple(fuzzy_set):
    a=np.sum(np.asarray([i*x for i,x in enumerate(fuzzy_set)]))
    b=np.sum(np.asarray([x for x in fuzzy_set]))
    beta=a/b
    index=np.round(beta)
    alpha=beta-index
    return (index,alpha)

def media_aritmetica(tuple_list):
    num=np.sum(np.asarray([tup[0]+tup[1] for tup in tuple_list]))
    beta=num/len(tuple_list)
    index=np.round(beta)
    alpha=beta-index
    return (index,alpha)

In [ ]:
def ExcelConcat():
  # Obtener una lista de todos los archivos de Excel en el directorio actual
  archivos_excel = glob.glob('*.xlsx')

  # Leer cada archivo de Excel en un DataFrame y agregarlo a una lista
  dfs = []
  for archivo in archivos_excel:
    df = pd.read_excel(archivo)
    dfs.append(df)

  # Concatenar los DataFrames en uno solo
  df_final = pd.concat(dfs,axis=1)
  return df_final

In [ ]:
def score(df:pd):
  score=df
  score['Clasificacion_Difusa']=score['Clasicacion_Difusa'].replace([4,3,2,1,0],['POS','POS','NEU','NEG','NEG'])
  score['clasificacion_pysentimiento']=score['clasificacion_pysentimiento'].replace([2,1,0],['POS','NEU','NEG'])
  score['Clasificacion_Vader']=score['Clasificacion_Vader'].replace([2,1,0],['POS','NEU','NEG'])
  score['Clasificacion_Asentiment']=score['Clasificacion_Asentiment'].replace([2,1,0],['POS','NEU','NEG'])
  score.to_excel('score.xlsx')


In [ ]:
def difuso_clasificator(data:str, ColumnName:str, C=False):
  PysentimentClasificator(data, ColumnName)
  VaderClasificator(data,ColumnName)
  AsentClasificator(data,ColumnName, C)
  var={}

  #Trabajar con excel y dataframe
  df = ExcelConcat()
  df_select=df.loc[:, ['clasificacion_pysentimiento', 'Clasificacion_Vader','Clasificacion_Asentiment']] #seleccionar columnas
  df2=df_select.replace(['POS','NEU','NEG'],[2,1,0]) #sustituir etiquetas por valores numericos

  polarity=generate_fuzzy_set(5) # la polaridad tiene un agranularidad de 5 (muy negativo (0), negativo(1),neutral(2),positivo(3), muy positivo(4))
  model=generate_fuzzy_set(3) # los modelos clasifican con 3 etiquetas
  #experts_votation=[2,2,2,2,2,2] # positivo, neutral, positivo
  #experts_votation = df2.iloc[15]
  #print(experts_votation[1])

  for i, row in df2.iterrows(): #recorrer cada fila del dataframe
    experts_votation = df2.iloc[i] #selcciona cada datos de la fila
    fuzzy_sets_2_tuple=[]
    for index in experts_votation: #selecciona la votacion de los expertos
      fuzzy_set=transform_to_fuzzy_set(model[index],polarity)
      fuzzy_sets_2_tuple.append(fuzzy_set_2_tuple(fuzzy_set))
    print(media_aritmetica(fuzzy_sets_2_tuple))
    var[i]=media_aritmetica(fuzzy_sets_2_tuple)[0] #guarda cada resultado

  df=pd.read_excel(data,sheet_name='Sheet1')
  difuso=pd.DataFrame( {'Clasicacion_Difusa': var})
  final=pd.concat([df,df2,difuso], axis=1)
  final.to_excel('score_diffuse.xlsx')
  score(final)


In [ ]:
def Metric(etiqueta:str, metric:str='ClassificationReport', sorter:str='difuse', ClassNumber:int=3 ):
  df=pd.read_excel('score_diffuse.xlsx', sheet_name='Sheet1')
  df[etiqueta]=df[etiqueta].replace([5,4,3,2,1],[2,2,1,0,0])
  df['Clasicacion_Difusa']=df['Clasicacion_Difusa'].replace([4,3,2,1,0],[2,2,1,0,0])


  def ConfusionMatrix(df:pd, etiqueta:str, sorter:str='difuse', ClassNumber:int=3):
    # Generar datos de ejemplo de etiquetas reales y predichas
    print('hola', ClassNumber)
    actual = df[etiqueta]
    if sorter=='pysentiment' :
        predicted = df['clasificacion_pysentimiento']
    elif sorter=='vader':
      predicted = df['Clasificacion_Vader']
    elif  sorter=='asent':
      predicted = df['Clasificacion_Asentiment']
    else:
        predicted = df['Clasicacion_Difusa']
    # Crear la matriz de confusión utilizando sklearn
    confusion_matrix = metrics.confusion_matrix(actual, predicted)
    # Crear la visualización de la matriz de confusión
    if ClassNumber==2:
      cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = ['negativo','positivo'])
    else:
      cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = ['negativo','neutral','positivo'])
    # Graficar la matriz de confusión
    cm_display.plot()
    plt.show()

  def ClassificationReport(df:pd, etiqueta:str, sorter='difuse', ClassNumber=3):
    actual = df[etiqueta]
    if sorter=='pysentiment' :
        predicted = df['clasificacion_pysentimiento']
    elif sorter=='vader':
      predicted = df['Clasificacion_Vader']
    elif  sorter=='asent':
      predicted = df['Clasificacion_Asentiment']
    else:
        predicted = df['Clasicacion_Difusa']

    if ClassNumber==2:
      target_names = ['class 0', 'class 2']
    else:
      target_names = ['class 0','class 1' 'class 2']
    print(classification_report(actual,predicted, target_names=target_names, digits=4))


  if metric=='ConfusionMatrix':
      ConfusionMatrix(df,etiqueta, sorter, ClassNumber)
  else:
    ClassificationReport(df, etiqueta,sorter, ClassNumber)



